# E0 · UCI HAR 행동 인식 실험

기존 CNN을 피험자 단위 검증 분할로 다시 평가해 비교 기준을 만듭니다.

**실행 상태:** 드라이브 원본의 실험 코드 셀에는 저장된 실행 출력이 없습니다. 아래 결과는 실행한 것으로 간주하지 않습니다. 순서대로 실행하고 검증 지표를 기록하세요.

**데이터:** `DATA_PATH`를 현재 Colab의 `DATA/UCI-HAR Dataset` 위치에 맞춰야 합니다. 공식 test를 이용한 반복 모델 선택은 피하세요.

**평가 주의:** 이 E 계열 코드는 공식 test 지표를 출력합니다. 과거 비교 기록으로 읽고, 이후 모델 선택에는 validation 지표를 사용하세요.



## 0. 라이브러리와 실험 설정

필요한 패키지를 불러오고 이 실험의 변경점과 실행 방법을 코드 주석에서 확인합니다.



In [ ]:
# ============================================================
# 실험 E0: 베이스라인 재현 (피험자 단위 검증)
# ------------------------------------------------------------
# 원본: CNN_UCI_HAR.ipynb
# 목적: 모델 구조와 학습 설정(20 에폭)은 원본 그대로 두고,
#       "평가 방식"만 바로잡아 이후 실험의 기준값을 다시 측정
#
# 변경 요약 (Ctrl+F 로 "[E0 변경]" 검색)
#   1) Drive 마운트 + 데이터 경로 수정, 자동 다운로드 코드 삭제
#   2) 피험자 번호 로드 → 피험자 5명을 통째로 검증셋으로 분리 (validation_split 대체)
#   3) SEED 를 0, 1, 2 로 바꿔 3번 실행
#   4) 결과 기록용 요약(SIT↔STAND 오류, 피험자별 정확도)과 학습 곡선 추가
#
# 실행 방법: SEED 값만 0 → 1 → 2 로 바꿔 3번 실행하고,
#           맨 아래 SUMMARY 한 줄을 결과표에 옮겨 적기
# ============================================================

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt   # [E0 변경] 학습 곡선을 그리기 위해 추가

from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# [E0 변경] import zipfile 삭제: 자동 다운로드 코드를 없앴으므로 필요 없음




## 1. Seed

난수 시드를 고정합니다. 시드를 바꿔 재실행하면 초기화에 따른 변동을 볼 수 있습니다.



In [ ]:
# ============================================================
# 1. Seed
# ============================================================

# [E0 변경] SEED = 42 → 0, 1, 2 로 바꿔가며 3번 실행
# 이유: 결과는 가중치 초기값과 데이터 섞는 순서에 따라 매번 조금씩 달라짐.
#       한 번의 결과(예: 91.21%)만으로는 실험 간 차이가 진짜인지 운인지 알 수 없으므로
#       3번의 평균 ± 표준편차로 비교함
SEED = 0

np.random.seed(SEED)
tf.random.set_seed(SEED)




## 2. Data path

Colab에서 Drive를 연결하고 UCI HAR 데이터의 위치를 확인합니다.



In [ ]:
# ============================================================
# 2. Data path
# ============================================================

# [E0 변경] Google Drive 마운트 추가
# 이유: 원본에는 drive.mount() 가 없어서 Drive 에 있는 데이터를 읽을 수 없음
from google.colab import drive
drive.mount("/content/drive")

# [E0 변경] 데이터 경로를 프로젝트 폴더로 수정하고, 자동 다운로드 코드 삭제
# 이유 1: 데이터가 이미 Drive 프로젝트 폴더에 있음 (폴더 이름도 "UCI-HAR Dataset", 하이픈)
# 이유 2: 원본의 다운로드 분기는 /content 에 압축을 풀지만 DATA_PATH 는 그대로라서
#         다운로드 분기로 가면 4번 로드 단계에서 파일을 못 찾는 문제가 있었음
DATA_PATH = "/content/drive/MyDrive/가천대학교/2026-2/1. 기계학습프로그래밍/2week/DATA/UCI-HAR Dataset"

assert os.path.exists(DATA_PATH), f"데이터 경로를 확인하세요: {DATA_PATH}"
print("Dataset:", DATA_PATH)




## 3. Signal names

입력으로 사용할 가속도계·자이로스코프 9개 채널의 파일명을 정의합니다.



In [ ]:
# ============================================================
# 3. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]




## 4. Load dataset

공식 train/test 분할의 센서 창, 행동 라벨, 피험자 번호를 읽습니다.



In [ ]:
# ============================================================
# 4. Load dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"

    y = np.loadtxt(y_path).astype(int)

    # label 1~6 -> 0~5
    y = y - 1

    # [E0 변경] 피험자 번호(1~30) 파일도 읽음
    # 이유: 검증셋을 "사람 단위"로 나누려면 각 윈도우가 누구의 데이터인지 알아야 함
    subject_path = f"{DATA_PATH}/{split}/subject_{split}.txt"

    subject = np.loadtxt(subject_path).astype(int)

    return X, y, subject   # [E0 변경] subject 를 함께 반환


# Official train/test split
X_train, y_train, s_train = load_data("train")   # [E0 변경] s_train(피험자 번호) 추가
X_test, y_test, s_test = load_data("test")       # [E0 변경] s_test(피험자 번호) 추가




## 4-1. Subject-wise validation split   [E0 추가]

피험자 5·11·17·23·29를 검증용으로 떼어 학습 피험자와 겹치지 않게 합니다.



In [ ]:
# ============================================================
# 4-1. Subject-wise validation split   [E0 추가]
# ============================================================

# [E0 변경] validation_split=0.2 대신, 피험자 5명의 데이터를 통째로 검증셋으로 분리
# 이유 1: train 파일은 피험자 번호 순서로 정렬되어 있음.
#         validation_split=0.2 는 "파일 끝 20% 행"을 떼어 가므로 검증셋이 피험자 27~30번이 되고,
#         27번은 일부가 학습 쪽에도 섞여 들어감
# 이유 2: 인접한 윈도우는 50% 겹침. 같은 사람이 학습과 검증에 동시에 있으면
#         거의 같은 데이터를 보고 채점하는 셈이라 검증 점수가 부풀려짐
# 이유 3: test 셋은 "처음 보는 사람 9명"이므로, 검증셋도 같은 조건(처음 보는 사람)이어야
#         검증 결과로 test 성능을 제대로 예상할 수 있음
# 선택 기준: train 피험자 21명 목록에서 4명 간격으로 5명 (앞으로 모든 실험에서 고정)
VAL_SUBJECTS = [5, 11, 17, 23, 29]

val_mask = np.isin(s_train, VAL_SUBJECTS)   # 검증 피험자의 윈도우이면 True

X_val, y_val = X_train[val_mask], y_train[val_mask]
X_train, y_train = X_train[~val_mask], y_train[~val_mask]   # ~ 는 True/False 를 뒤집음


print("\nDataset shape")
print("X_train:", X_train.shape, "(16 subjects)")
print("X_val  :", X_val.shape, "subjects:", VAL_SUBJECTS)
print("X_test :", X_test.shape, "(9 subjects)")




## 5. Normalization

학습 데이터의 평균·표준편차만 사용해 검증·테스트 입력을 변환합니다.



In [ ]:
# ============================================================
# 5. Normalization
# ============================================================

# [E0 변경] 이 단계를 검증셋 분리 "뒤"에 둠
# 이유: 평균/표준편차를 학습 데이터(16명)로만 계산해야 검증·테스트 데이터 정보가 섞이지 않음
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = (X_train - mean) / (std + 1e-8)
X_val = (X_val - mean) / (std + 1e-8)       # [E0 변경] 검증셋도 train 통계로 정규화
X_test = (X_test - mean) / (std + 1e-8)




## 6. One-hot encoding

정수 행동 라벨을 6개 범주의 원핫 벡터로 바꿉니다.



In [ ]:
# ============================================================
# 6. One-hot encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

# [E0 변경] 검증셋 라벨도 one-hot 으로 변환
# 이유: categorical_crossentropy 손실은 정답을 one-hot 형태로 받기 때문
y_val_onehot = to_categorical(
    y_val,
    num_classes=n_outputs
)

y_test_onehot = to_categorical(
    y_test,
    num_classes=n_outputs
)




## 7. CNN Model

이 실험의 CNN 또는 CNN+LSTM 구조를 정의합니다. 비교 대상과 달라진 층은 코드 주석에 표시했습니다.



In [ ]:
# ============================================================
# 7. CNN Model
# ============================================================

inputs = Input(shape=(128, 9))


conv1 = Conv1D(
    filters=32,
    kernel_size=3,
    activation="relu"
)(inputs)

pool1 = MaxPooling1D(
    pool_size=2
)(conv1)


conv2 = Conv1D(
    filters=64,
    kernel_size=3,
    activation="relu"
)(pool1)

pool2 = MaxPooling1D(
    pool_size=2
)(conv2)


flat = Flatten()(pool2)


dense = Dense(
    128,
    activation="relu"
)(flat)


outputs = Dense(
    n_outputs,
    activation="softmax"
)(dense)




## 8. Model

모델을 만들고 손실 함수, 옵티마이저, 평가 지표를 설정합니다.



In [ ]:
# ============================================================
# 8. Model
# ============================================================

model = Model(
    inputs=inputs,
    outputs=outputs
)


model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


model.summary()




## 9. Train

학습 피험자의 데이터로 모델을 훈련하고 학습 이력을 저장합니다.



In [ ]:
# ============================================================
# 9. Train
# ============================================================

history = model.fit(
    X_train,
    y_train_onehot,
    epochs=20,
    batch_size=64,
    # [E0 변경] validation_split=0.2 → validation_data=(X_val, y_val_onehot)
    # 이유: 4-1 에서 만든 피험자 단위 검증셋을 사용 (파일 끝 20% 를 자르지 않음)
    validation_data=(X_val, y_val_onehot),
    shuffle=True,
    verbose=1
)




## 10. Prediction

학습한 모델로 공식 test 창의 행동을 예측합니다. 이후의 모델 선택에는 이 점수를 쓰지 않습니다.



In [ ]:
# ============================================================
# 10. Prediction
# ============================================================

y_prob = model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    y_prob,
    axis=1
)




## 11. Evaluation

공식 test의 정확도와 Macro Precision·Recall·F1을 계산합니다.



In [ ]:
# ============================================================
# 11. Evaluation
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n==============================")
print("TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")




## 12. Classification Report

클래스별 precision·recall·F1을 읽어 어떤 행동이 어려운지 확인합니다.



In [ ]:
# ============================================================
# 12. Classification Report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]


print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)




## 13. Confusion Matrix

혼동 행렬을 그려 실제 행동과 예측 행동이 어디서 엇갈리는지 확인합니다.



In [ ]:
# ============================================================
# 13. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)


print("\n==============================")
print("CONFUSION MATRIX")
print("==============================")

print(cm)




## 14. Per-subject accuracy & Summary   [E0 추가]

공식 test 피험자별 정확도와 다음 비교에 옮겨 적을 요약값을 출력합니다.



In [ ]:
# ============================================================
# 14. Per-subject accuracy & Summary   [E0 추가]
# ============================================================

# [E0 변경] 테스트 피험자별 정확도 출력
# 이유: 전체 정확도가 같아도 특정 사람(예: 9, 10번)에게만 크게 틀릴 수 있음.
#       "처음 보는 사람"에 대한 일반화가 목표이므로 가장 낮은 피험자도 함께 확인
print("\n==============================")
print("PER-SUBJECT ACCURACY (test)")
print("==============================")

subject_acc = {}

for s in np.unique(s_test):

    m = (s_test == s)

    subject_acc[s] = accuracy_score(y_test[m], y_pred[m])

    print(f"subject {s:2d}: {subject_acc[s]:.4f}")

worst_subject = min(subject_acc, key=subject_acc.get)


# [E0 변경] 결과표에 옮겨 적을 값을 한 줄로 출력
# 이유: 정확도만으로는 "무엇이" 좋아졌는지 알 수 없음. 핵심 문제별 오류 수를 매 실험 같은 방식으로 기록
#   - SIT<->STAND : 혼동 행렬 [3,4] + [4,3]. 원본 오류의 65%를 차지한 가장 큰 문제
#   - dynamic     : 걷기/계단 오르기/계단 내리기 3개 클래스끼리 서로 틀린 개수
#   - best epoch  : val_loss 가 가장 낮았던 에폭. 이후 계속 학습했다면 과적합 구간
sit_stand_err = cm[3, 4] + cm[4, 3]

dynamic_err = cm[0:3, 0:3].sum() - np.trace(cm[0:3, 0:3])

best_epoch = int(np.argmin(history.history["val_loss"])) + 1

print("\n==============================")
print("SUMMARY (결과표에 기록)")
print("==============================")

print(
    f"E0 | seed {SEED} | acc {accuracy:.4f} | macro F1 {f1:.4f} | "
    f"SIT<->STAND {sit_stand_err} | dynamic {dynamic_err} | "
    f"worst subject {worst_subject} ({subject_acc[worst_subject]:.4f}) | "
    f"epochs {len(history.history['val_loss'])} (best {best_epoch}) | "
    f"params {model.count_params():,}"
)




## 15. Learning curve   [E0 추가]

학습·검증 곡선을 그려 에폭에 따른 지표 변화를 확인합니다.



In [ ]:
# ============================================================
# 15. Learning curve   [E0 추가]
# ============================================================

# [E0 변경] loss / accuracy 학습 곡선 추가
# 이유: val_loss 가 몇 에폭부터 다시 오르는지(과적합이 시작되는 시점)를 눈으로 확인하기 위함
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("epoch")
plt.title(f"E0 loss (seed {SEED})")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.xlabel("epoch")
plt.title(f"E0 accuracy (seed {SEED})")
plt.legend()

plt.tight_layout()
plt.show()
